# Lesson 3.6 — Integrating Claude Code into CI/CD Pipelines

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Run Claude Code non-interactively with -p (or --print) to avoid pipeline hangs
- Use --output-format json and --json-schema for machine-parseable findings
- Provide project context via CLAUDE.md for CI-invoked reviews and test generation
- Use an independent review instance and report only new/unaddressed issues

**You will produce:** A CI review script using -p + --output-format json and a JSON schema for PR findings.

## Concepts

Running Claude Code in CI:

- **`-p` / `--print`** runs non-interactively: it processes the prompt, writes to stdout, and exits. Sample Question 10: a bare `claude "..."` hangs waiting for input; the fix is `-p`.
- **`--output-format json`** with **`--json-schema`** enforces structured output you can post as inline PR comments.
- **CLAUDE.md** supplies project context (testing standards, fixture conventions, review criteria) to CI-invoked Claude Code, improving quality and reducing low-value output.
- **Session isolation:** the same session that generated code is weaker at reviewing it. Use an **independent review instance**; include prior findings so it reports only **new or still-unaddressed** issues, and provide existing tests so generation avoids duplicates.

> **Reference:** [Anthropic - Claude Code CLI / headless](https://docs.anthropic.com/en/docs/claude-code/cli-reference)

In [ ]:
import os, pathlib
ROOT = pathlib.Path('cc_project'); ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
(pathlib.Path('.claude') / 'rules').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'commands').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'skills').mkdir(parents=True, exist_ok=True)
print('project root:', os.getcwd())

## Code-Along: Step 1 — A non-interactive CI command

The `-p` flag is what prevents the hang from Sample Question 10.

In [ ]:
def ci_review_command(prompt, schema_path='finding.schema.json'):
    return (f"claude -p {prompt!r} "
            f"--output-format json --json-schema {schema_path}")

print(ci_review_command('Review the diff for security issues'))

## Code-Along: Step 2 — Define a findings JSON schema

Machine-parseable findings enable automated inline PR comments.

In [ ]:
import json
finding_schema = {
    "type": "object",
    "properties": {
        "findings": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "issue": {"type": "string"},
                "severity": {"type": "string",
                    "enum": ["low", "medium", "high", "critical"]},
                "suggested_fix": {"type": "string"}
            },
            "required": ["location", "issue", "severity"]
        }}
    },
    "required": ["findings"]
}
pathlib.Path('finding.schema.json').write_text(json.dumps(finding_schema, indent=2),
                                                encoding='utf-8')
print('wrote finding.schema.json')

## Code-Along: Step 3 — Run the review (guarded) via the CLI

This cell shells out to Claude Code if installed; otherwise it prints the command. It never hangs because of `-p`.

In [ ]:
import shutil, subprocess
cmd = ci_review_command('Review staged changes; report only NEW issues')
if shutil.which('claude'):
    try:
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120)
        print(out.stdout[:500] or out.stderr[:500])
    except Exception as e:
        print('CLI call skipped:', e)
else:
    print('claude CLI not installed - would run:')
    print(cmd)

## Exercise

Write a `rereview_command(prior_findings_path)` that includes prior findings so the independent instance reports only new or unaddressed issues. Add a `test_gen_command` that passes existing test files in context to avoid duplicate scenarios, and note in a comment which CLAUDE.md sections improve test quality.

In [ ]:
# TODO: rereview_command(prior_findings_path) -> claude -p ... --output-format json
# TODO: test_gen_command(existing_tests) that avoids duplicate test scenarios

## Key Takeaways

- Use -p/--print to run Claude Code non-interactively so CI jobs don't hang.
- --output-format json + --json-schema produce machine-parseable PR findings.
- CLAUDE.md feeds CI reviews/test-gen the project standards and fixtures.
- Use an independent review instance; report only new/unaddressed issues and avoid duplicate tests.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.